# Day 4: Evaluating a RAG pipeline

## Introduction
This notebook follows the lesson's original code and execution order. It loads a labeled question set, inspects one example, evaluates retrieval, and then generates and judges an answer. Only explanatory comments and markdown have been added; the executable statements and default scoring behavior are unchanged.

**Two evaluation questions:** did retrieval find the expected evidence, and did generation produce a correct, complete, relevant answer? These are related but separate checks.

### Before running
- Use the project Python environment with the kernel working directory set to `my-work/Week 5`, so the local `evaluation` and `implementation` imports resolve correctly.
- Run [Day 2](./2-day2.ipynb) first to create the local MiniLM Chroma collection. The current helper uses normalized `sentence-transformers/all-MiniLM-L6-v2` embeddings and collection `insurellm_day2_minilm_tokens200_overlap40_v1`.
- Configure API credentials in the repository `.env` or environment. Answer evaluation runs directly, as in the lesson: it makes one generation call and one judging call, both potentially billable. There is no opt-in flag in this version.
- Run cells in order. Restart the kernel after changing imported Python modules; rerunning an import statement alone does not reload cached definitions.

### Supporting modules
| Module | Responsibility |
| --- | --- |
| [evaluation/test.py](./evaluation/test.py) | Load JSONL records as `TestQuestion` objects. |
| [evaluation/eval.py](./evaluation/eval.py) | Calculate retrieval metrics and compare generated answers with references using an LLM judge. |
| [implementation/answer.py](./implementation/answer.py) | Retrieve local chunks and return an answer together with its generation context. |

The helper descriptions below reflect the current local implementation. This documentation pass does not modify those modules.

## 1. Load and inspect the test questions

`load_tests()` reads `evaluation/tests.jsonl` relative to the loader module and creates validated `TestQuestion` objects. Each record contains `question`, `category`, `reference_answer`, and `keywords`. Validation checks the declared field types; it does not establish the factual correctness of the reference labels.

Loading the list and displaying its length does not evaluate every question. The lesson selects only `tests[0]` for the evaluations later in this notebook. This assumes the dataset is nonempty.

In [8]:
# Import the local dataset loader and TestQuestion schema.
from evaluation import test

In [9]:
# Load labeled questions; this does not call the answer model or judge.
tests = test.load_tests()

In [10]:
# Jupyter displays the number of loaded questions, not a performance score.
len(tests)

150

In [11]:
# Use the same first question for retrieval and answer evaluation.
example = tests[0]
# Keywords describe expected retrieved evidence; the reference is the target answer.
print(example.question)
print(example.category)
print(example.reference_answer)
print(example.keywords)


Who won the prestigious IIOTY award in 2023?
direct_fact
Maxine Thompson won the prestigious Insurellm Innovator of the Year (IIOTY) award in 2023.
['Maxine', 'Thompson', 'IIOTY']


### Inspect category coverage
`Counter` groups questions by their category labels. This describes the dataset's composition, not the system's accuracy. An overall score across a larger benchmark could be dominated by its most common category.

In [12]:
from collections import Counter
# Count how many questions belong to each category; no tests are executed here.
count = Counter([t.category for t in tests])
count

Counter({'direct_fact': 70,
         'temporal': 20,
         'spanning': 20,
         'comparative': 10,
         'numerical': 10,
         'relationship': 10,
         'holistic': 10})

### Dataset summary
- `tests` holds all loaded questions; `example` refers to just the first one.
- The reference answer is used for answer judging; keywords are used for retrieval matching.
- Category counts help identify coverage gaps but are not evaluation results.

## 2. Evaluate retrieval quality

Importing the evaluator also loads its answer helper, which initializes MiniLM embeddings, the local Chroma collection, and the chat client. The first embedding-model load may download files. The retrieval evaluation itself searches local chunks without calling the generation model or judge.

`evaluate_retrieval(example)` uses the helper's default cutoff of 10 chunks. In the current local helper, all metrics use the same top-k result set and match keywords with **case-insensitive substring searches**, not semantic or whole-word matching.

| Returned field | Meaning in this implementation |
| --- | --- |
| `mrr` | Mean reciprocal rank across this question's keywords: a keyword first found at rank 1 contributes 1, rank 2 contributes 0.5, and an absent keyword contributes 0. This is not an average across all test questions. |
| `ndcg` | Per-keyword binary relevance discounted by `log2(rank + 1)`, normalized against the best ordering of those same returned relevance labels, then averaged across keywords. |
| `keywords_found` / `total_keywords` | How many expected keywords appear in at least one returned chunk. |
| `keyword_coverage` | Percentage of expected keywords found, on a 0-100 scale. |

MRR and nDCG are on a 0-1 scale. This nDCG measures ranking within the retrieved set, not recall against all relevant corpus documents. The helper returns zeros when the keyword list is empty; that is an uninformative case, not evidence that retrieval failed.

In [13]:
# Load the retrieval scorer and the separate generation-plus-judge evaluator.
from evaluation.eval import evaluate_retrieval, evaluate_answer

In [14]:
# Score only the selected question; Jupyter displays a RetrievalEval object.
evaluate_retrieval(example)

RetrievalEval(mrr=0.6666666666666666, ndcg=0.5543697530557483, keywords_found=2, total_keywords=3, keyword_coverage=66.66666666666666)

### Retrieval summary
High coverage means the expected strings were found somewhere in the returned chunks; higher reciprocal rank favors finding them earlier. Neither guarantees a complete or correct answer. Inspect evidence when an important surname, date, or number is missing. Compare retrieval configurations using the same questions, labels, corpus, and cutoff.

## 3. Generate an answer and judge its quality

`evaluate_answer(example)` runs synchronously. First, the RAG helper retrieves context and generates an answer. Then a separate LLM call judges that answer against the reference. The function returns three values:

| Lesson variable | Value |
| --- | --- |
| `eval` | `AnswerEval` object containing feedback and numeric scores. This variable name is preserved from the lesson; it shadows Python's built-in `eval` function. |
| `answer` | Generated answer text. |
| `chunks` | Exact retrieved documents supplied to the generation model, including their metadata. |

**Cost and execution:** running the next cell makes the generation and judging requests immediately. Rerunning it makes fresh requests. The following display cells only inspect the stored result and do not call the model again.

**Judge limitation:** the judge receives the question, generated answer, and reference answer, but not `chunks`. Its feedback therefore does not directly verify whether the answer is faithful to retrieved sources. The variables `answer` and `chunks` remain available for manual inspection even though the original lesson's display cells focus on the evaluation object.

In [15]:
# Two model calls: generate the RAG answer, then judge it against the reference.
# Unpack feedback/scores, generated text, and the generation context in that order.
eval, answer, chunks = evaluate_answer(example)

In [16]:
# Display the stored feedback and scores; this does not make another API call.
eval

AnswerEval(feedback='The answer identifies Maxine as the winner but omits her surname, Thompson. It also expands the award name differently from the reference, though it clearly points to the same award and year. It directly answers the question, but is not fully complete or perfectly precise.', accuracy=3.0, completeness=3.0, relevance=5.0)

### Read the judge's feedback and scores
The judge's rubric requests scores from 1 to 5:
- **Accuracy:** factual agreement with the reference; the prompt instructs the judge to assign 1 for a wrong answer.
- **Completeness:** whether all information needed from the reference is included.
- **Relevance:** whether the answer addresses the question without unnecessary additions.

The current `AnswerEval` schema requires numeric fields but does not enforce the 1-5 range. These are model judgments, not calibrated probabilities or guaranteed facts. Read feedback as well as numbers, and verify questionable judgments against the reference and supporting evidence. The code is left unchanged.

In [17]:
# Print the stored explanation, followed by accuracy, completeness, and relevance.
print(eval.feedback)
print(eval.accuracy)
print(eval.completeness)
print(eval.relevance)

The answer identifies Maxine as the winner but omits her surname, Thompson. It also expands the award name differently from the reference, though it clearly points to the same award and year. It directly answers the question, but is not fully complete or perfectly precise.
3.0
3.0
5.0


### Answer evaluation summary
- Retrieval scores measure keyword evidence; answer scores compare generated text with a reference.
- An answer can be relevant yet incomplete, or contain the right keywords without being correct.
- If the answer misses a fact, inspect whether it was present in `chunks` before blaming generation.
- Loading the full dataset does not make this a full benchmark: only `example` has been evaluated.

## Conclusion
This lesson demonstrates a two-stage RAG evaluation workflow: check retrieval evidence, then judge the generated answer against an expected answer. Keeping those stages separate helps identify whether a poor result comes from missing context, generation, or limitations in the test labels and judge.

### Reference checklist
1. Confirm the local environment, Day 2 collection, credentials, and fresh kernel state.
2. Inspect the selected question, expected keywords, and reference answer.
3. Read retrieval ranking and coverage together, remembering that matching is substring-based.
4. Run answer evaluation knowing that it makes two model requests.
5. Review feedback, answer text, and source chunks instead of relying on scores alone.
6. Treat one question as an illustrative smoke test, not proof of system-wide performance; broader claims require multiple questions and category-level analysis.

**Preservation note:** the original executable statements, variable names, cell order, and direct scoring behavior have been retained. No flags, regression tests, automatic reloads, or helper changes were introduced in this documentation pass.